# Cross-Platform Demo: Paimon + Iceberg

## What is Paimon's Iceberg Compatibility?
- Paimon can write Iceberg-compatible metadata
- The SAME physical table can be queried via both:
  - Paimon catalog (native Paimon features)
  - Iceberg catalog (standard Iceberg tools)

## This demo will demonstrate:
1. Creating a Paimon table with Iceberg compatibility
2. Querying the same table via both catalogs
3. Drop table behavior across catalogs
4. ALTER TABLE compatibility
5. REST Catalog integration

## Setup: Import libraries and helper functions

In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException
from py4j.protocol import Py4JJavaError

In [2]:
def print_query(sql_query, description=None):
    """Print SQL query with optional description."""
    if description:
        print(f"\n   📝 {description}")
    print(f"   SQL: {sql_query}")


def find_jar_files(base_dir):
    """Find Paimon, Iceberg, and Paimon-Iceberg JAR files in the jars directory."""
    jars_dir = os.path.join(base_dir, "jars")
    paimon_jar = None
    iceberg_jar = None
    paimon_iceberg_jar = None
    
    for file in os.listdir(jars_dir):
        if "paimon-spark" in file and file.endswith('.jar'):
            paimon_jar = os.path.join(jars_dir, file)
        elif "iceberg-spark-runtime" in file and file.endswith('.jar'):
            iceberg_jar = os.path.join(jars_dir, file)
        elif "paimon-iceberg" in file and file.endswith('.jar'):
            paimon_iceberg_jar = os.path.join(jars_dir, file)
    
    if not paimon_jar or not iceberg_jar:
        raise FileNotFoundError("JAR files not found. Please run setup.sh first.")
    
    return paimon_jar, iceberg_jar, paimon_iceberg_jar

In [3]:
def create_spark_session(base_dir):
    """Create Spark session with both Paimon and Iceberg catalogs configured."""
    paimon_jar, iceberg_jar, paimon_iceberg_jar = find_jar_files(base_dir)
    
    paimon_warehouse = f"file://{base_dir}/warehouse/paimon"
    iceberg_warehouse = f"file://{base_dir}/warehouse/paimon/iceberg"
    
    print("🔧 Configuring Spark with dual catalogs")
    print(f"   Paimon JAR: {os.path.basename(paimon_jar)}")
    print(f"   Iceberg JAR: {os.path.basename(iceberg_jar)}")
    if paimon_iceberg_jar:
        print(f"   Paimon-Iceberg JAR: {os.path.basename(paimon_iceberg_jar)} (REST Catalog support)")
    print(f"   Paimon Warehouse: {paimon_warehouse}")
    print(f"   Iceberg Warehouse: {iceberg_warehouse}")
    
    # Build JAR list
    jars_list = [paimon_jar, iceberg_jar]
    if paimon_iceberg_jar:
        jars_list.append(paimon_iceberg_jar)
    jars_string = ",".join(jars_list)
    
    spark = SparkSession.builder \
        .appName("Cross-Platform Demo") \
        .config("spark.jars", jars_string) \
        .config("spark.sql.catalog.paimon_catalog", "org.apache.paimon.spark.SparkCatalog") \
        .config("spark.sql.catalog.paimon_catalog.warehouse", paimon_warehouse) \
        .config("spark.sql.catalog.iceberg_catalog", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.iceberg_catalog.type", "hadoop") \
        .config("spark.sql.catalog.iceberg_catalog.warehouse", iceberg_warehouse) \
        .config("spark.sql.catalog.iceberg_catalog.cache-enabled", "false") \
        .config("spark.sql.extensions",
                "org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions,"
                "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("WARN")
    return spark

In [4]:
# Get base directory and create Spark session
base_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
spark = create_spark_session(base_dir)
print("✅ Spark session with dual catalogs created successfully!")

🔧 Configuring Spark with dual catalogs
   Paimon JAR: paimon-spark-3.4-1.3.0.jar
   Iceberg JAR: iceberg-spark-runtime-3.4_2.12-1.10.0.jar
   Paimon-Iceberg JAR: paimon-iceberg-1.3.0.jar (REST Catalog support)
   Paimon Warehouse: file:///Users/yerachmielfeltzman/projects/personal/apache-paimon-demo/warehouse/paimon
   Iceberg Warehouse: file:///Users/yerachmielfeltzman/projects/personal/apache-paimon-demo/warehouse/paimon/iceberg


26/05/13 14:11:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Spark session with dual catalogs created successfully!


---
# DEMO 1: Cross-Platform Query

## Key Concept: One table, two catalog interfaces
- Paimon writes Iceberg metadata alongside its own
- Enables interoperability between ecosystems

## Step 1: Create Paimon table with Iceberg compatibility

**Key Concept**: Set `'metadata.iceberg.storage' = 'hadoop-catalog'`

In [5]:
table_name = "paimon_catalog.`default`.cities"

create_sql = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        country STRING,
        name STRING
    )
    TBLPROPERTIES (
        'metadata.iceberg.storage' = 'hadoop-catalog'
    )
"""
print_query(create_sql.strip(), "Creating table with Iceberg compatibility")
spark.sql(create_sql)
print("✅ Table created")
print("💡 Paimon will write Iceberg metadata to <warehouse>/iceberg/")


   📝 Creating table with Iceberg compatibility
   SQL: CREATE TABLE IF NOT EXISTS paimon_catalog.`default`.cities (
        country STRING,
        name STRING
    )
    TBLPROPERTIES (
        'metadata.iceberg.storage' = 'hadoop-catalog'
    )
✅ Table created
💡 Paimon will write Iceberg metadata to <warehouse>/iceberg/


## Step 2: Insert sample data via Paimon catalog

In [6]:
insert_sql = f"""
    INSERT INTO {table_name} VALUES 
        ('usa', 'new york'),
        ('germany', 'berlin'),
        ('usa', 'chicago'),
        ('germany', 'hamburg')
"""
print_query(insert_sql.strip(), "Inserting 4 rows of sample data")
spark.sql(insert_sql)
print("✅ Data inserted")


   📝 Inserting 4 rows of sample data
   SQL: INSERT INTO paimon_catalog.`default`.cities VALUES 
        ('usa', 'new york'),
        ('germany', 'berlin'),
        ('usa', 'chicago'),
        ('germany', 'hamburg')


✅ Data inserted


## Step 3: Query via Paimon catalog (native)

In [7]:
query_paimon = f"SELECT * FROM {table_name} WHERE country = 'germany'"
print_query(query_paimon, "Querying German cities via Paimon")
result_paimon = spark.sql(query_paimon)
result_paimon.show()
paimon_count = result_paimon.count()
print(f"✅ Found {paimon_count} German cities via Paimon")


   📝 Querying German cities via Paimon
   SQL: SELECT * FROM paimon_catalog.`default`.cities WHERE country = 'germany'
+-------+-------+
|country|   name|
+-------+-------+
|germany| berlin|
|germany|hamburg|
|germany| berlin|
|germany|hamburg|
|germany| berlin|
|germany|hamburg|
|germany| berlin|
|germany|hamburg|
|germany| berlin|
|germany|hamburg|
|germany| berlin|
|germany|hamburg|
+-------+-------+

✅ Found 12 German cities via Paimon


## Step 4: Query the SAME table via Iceberg catalog

**Key Concept**: Same table, different catalog!

In [8]:
query_iceberg = "SELECT * FROM iceberg_catalog.`default`.cities WHERE country = 'germany'"
print_query(query_iceberg, "Querying via Iceberg catalog (cross-platform)")
print("📚 Notice: Same table, different catalog!")
result_iceberg = spark.sql(query_iceberg)
result_iceberg.show()
iceberg_count = result_iceberg.count()
print(f"✅ Found {iceberg_count} German cities via Iceberg")


   📝 Querying via Iceberg catalog (cross-platform)
   SQL: SELECT * FROM iceberg_catalog.`default`.cities WHERE country = 'germany'
📚 Notice: Same table, different catalog!
+-------+-------+
|country|   name|
+-------+-------+
|germany| berlin|
|germany| berlin|
|germany| berlin|
|germany| berlin|
|germany| berlin|
|germany| berlin|
|germany|hamburg|
|germany|hamburg|
|germany|hamburg|
|germany|hamburg|
|germany|hamburg|
|germany|hamburg|
+-------+-------+

✅ Found 12 German cities via Iceberg


## Result: Cross-platform compatibility confirmed!

In [9]:
print("✨ SUCCESS! The same physical table queried via both catalogs!")
print(f"   • Paimon catalog: {paimon_count} rows")
print(f"   • Iceberg catalog: {iceberg_count} rows")
print("💡 This enables interoperability between Paimon and Iceberg ecosystems")

✨ SUCCESS! The same physical table queried via both catalogs!
   • Paimon catalog: 12 rows
   • Iceberg catalog: 12 rows
💡 This enables interoperability between Paimon and Iceberg ecosystems


---
# DEMO 2: Drop Table Behavior Test

## Question: What happens when you drop a Paimon table?
- Does the Iceberg mirror get dropped automatically?
- Or does it persist (requiring manual cleanup)?

## Step 1: Clean up existing tables

In [10]:
table_name = "paimon_catalog.`default`.test_drop"
iceberg_table_name = "iceberg_catalog.`default`.test_drop"

drop_sql = f"DROP TABLE IF EXISTS {iceberg_table_name}"
print_query(drop_sql, "Dropping Iceberg table if exists")
try:
    spark.sql(drop_sql)
except Exception as e:
    print(f"   ⚠️  {str(e)}")

drop_sql = f"DROP TABLE IF EXISTS {table_name}"
print_query(drop_sql, "Dropping Paimon table if exists")
spark.sql(drop_sql)
print("✅ Cleanup complete")


   📝 Dropping Iceberg table if exists
   SQL: DROP TABLE IF EXISTS iceberg_catalog.`default`.test_drop

   📝 Dropping Paimon table if exists
   SQL: DROP TABLE IF EXISTS paimon_catalog.`default`.test_drop
✅ Cleanup complete


## Step 2: Create Paimon table with Iceberg compatibility

In [11]:
create_sql = f"""
    CREATE TABLE {table_name} (
        id INT,
        name STRING
    )
    TBLPROPERTIES (
        'metadata.iceberg.storage' = 'hadoop-catalog'
    )
"""
print_query(create_sql.strip(), "Creating table with Iceberg compatibility")
spark.sql(create_sql)
print("✅ Table created")


   📝 Creating table with Iceberg compatibility
   SQL: CREATE TABLE paimon_catalog.`default`.test_drop (
        id INT,
        name STRING
    )
    TBLPROPERTIES (
        'metadata.iceberg.storage' = 'hadoop-catalog'
    )
✅ Table created


## Step 3: Insert test data

In [12]:
insert_sql = f"""
    INSERT INTO {table_name} VALUES 
        (1, 'Alice'),
        (2, 'Bob'),
        (3, 'Charlie')
"""
print_query(insert_sql.strip(), "Inserting 3 test rows")
spark.sql(insert_sql)
print("✅ Data inserted")


   📝 Inserting 3 test rows
   SQL: INSERT INTO paimon_catalog.`default`.test_drop VALUES 
        (1, 'Alice'),
        (2, 'Bob'),
        (3, 'Charlie')
✅ Data inserted


## Step 4: Verify table is accessible via BOTH catalogs

In [13]:
query_paimon = f"SELECT COUNT(*) as count FROM {table_name}"
print_query(query_paimon, "Count via Paimon catalog")
paimon_count = spark.sql(query_paimon).collect()[0]['count']
print(f"✅ Paimon sees {paimon_count} rows")

query_iceberg = f"SELECT COUNT(*) as count FROM {iceberg_table_name}"
print_query(query_iceberg, "Count via Iceberg catalog")
iceberg_count = spark.sql(query_iceberg).collect()[0]['count']
print(f"✅ Iceberg sees {iceberg_count} rows")


   📝 Count via Paimon catalog
   SQL: SELECT COUNT(*) as count FROM paimon_catalog.`default`.test_drop
✅ Paimon sees 3 rows

   📝 Count via Iceberg catalog
   SQL: SELECT COUNT(*) as count FROM iceberg_catalog.`default`.test_drop
✅ Iceberg sees 3 rows


## Step 5: Drop the Paimon table

In [14]:
drop_sql = f"DROP TABLE {table_name}"
print_query(drop_sql, "Dropping Paimon table")
spark.sql(drop_sql)
print("✅ Paimon table dropped")


   📝 Dropping Paimon table
   SQL: DROP TABLE paimon_catalog.`default`.test_drop
✅ Paimon table dropped


## Step 6: Test if Iceberg table still exists after Paimon drop

In [15]:
query_test = f"SELECT * FROM {iceberg_table_name}"
print_query(query_test, "Querying Iceberg table after Paimon drop")

try:
    result = spark.sql(query_test)
    result.show()
    count = result.count()
    print(f"\n⚠️  RESULT: Iceberg table STILL EXISTS with {count} rows!")
    print("\n📚 CONCLUSION:")
    print("   • Dropping Paimon table does NOT drop Iceberg metadata")
    print("   • The Iceberg table persists (potentially broken)")
    print("   💡 Recommendation: Manually drop both tables when cleaning up")
except Exception as e:
    error_msg = str(e)
    if "not found" in error_msg.lower() or "does not exist" in error_msg.lower():
        print(f"\n✅ RESULT: Iceberg table was automatically removed!")
        print("\n📚 CONCLUSION:")
        print("   • Dropping Paimon table also removes Iceberg metadata")
        print("   • This is clean behavior for mirrored tables")
    else:
        print(f"\n⚠️  RESULT: Iceberg table exists but is broken!")
        print(f"   Error: {error_msg[:150]}...")
        print("\n📚 CONCLUSION:")
        print("   • Iceberg metadata exists but data files were deleted")
        print("   • This creates an inconsistent state")


   📝 Querying Iceberg table after Paimon drop
   SQL: SELECT * FROM iceberg_catalog.`default`.test_drop

✅ RESULT: Iceberg table was automatically removed!

📚 CONCLUSION:
   • Dropping Paimon table also removes Iceberg metadata
   • This is clean behavior for mirrored tables


26/05/13 14:12:05 ERROR BaseReader: Error reading file(s): file:/Users/yerachmielfeltzman/projects/personal/apache-paimon-demo/warehouse/paimon/default.db/test_drop/bucket-0/data-5bf5dec3-9dd4-44ee-af24-83d2533e0e57-0.parquet
org.apache.iceberg.exceptions.NotFoundException: File does not exist: file:/Users/yerachmielfeltzman/projects/personal/apache-paimon-demo/warehouse/paimon/default.db/test_drop/bucket-0/data-5bf5dec3-9dd4-44ee-af24-83d2533e0e57-0.parquet
	at org.apache.iceberg.hadoop.HadoopInputFile.lazyStat(HadoopInputFile.java:164)
	at org.apache.iceberg.hadoop.HadoopInputFile.getStat(HadoopInputFile.java:200)
	at org.apache.iceberg.parquet.ParquetIO.file(ParquetIO.java:51)
	at org.apache.iceberg.parquet.ReadConf.newReader(ReadConf.java:194)
	at org.apache.iceberg.parquet.ReadConf.<init>(ReadConf.java:76)
	at org.apache.iceberg.parquet.VectorizedParquetReader.init(VectorizedParquetReader.java:90)
	at org.apache.iceberg.parquet.VectorizedParquetReader.iterator(VectorizedParquetRea

## Step 7: Final cleanup

In [16]:
cleanup_sql = f"DROP TABLE IF EXISTS {iceberg_table_name}"
print_query(cleanup_sql, "Cleaning up Iceberg table")
try:
    spark.sql(cleanup_sql)
    print("✅ Iceberg table cleaned up")
except Exception as e:
    print(f"   ⚠️  {str(e)}")
    print("✅ No Iceberg table to clean up")


   📝 Cleaning up Iceberg table
   SQL: DROP TABLE IF EXISTS iceberg_catalog.`default`.test_drop
✅ Iceberg table cleaned up


---
# DEMO 3: ALTER TABLE Compatibility Test

## Question: Can we add Iceberg compatibility to an existing table?
- Create table WITHOUT Iceberg property
- Use ALTER TABLE to add the property
- Check if old data becomes visible via Iceberg

## Step 1: Clean up (drop tables if they exist)

In [17]:
table_name = "paimon_catalog.`default`.cities2"
iceberg_table_name = "iceberg_catalog.`default`.cities2"

drop_sql = f"DROP TABLE IF EXISTS {iceberg_table_name}"
print_query(drop_sql, "Dropping Iceberg table if exists")
try:
    spark.sql(drop_sql)
except Exception as e:
    print(f"   ⚠️  {str(e)}")

drop_sql = f"DROP TABLE IF EXISTS {table_name}"
print_query(drop_sql, "Dropping Paimon table if exists")
spark.sql(drop_sql)
print("✅ Cleanup complete")


   📝 Dropping Iceberg table if exists
   SQL: DROP TABLE IF EXISTS iceberg_catalog.`default`.cities2

   📝 Dropping Paimon table if exists
   SQL: DROP TABLE IF EXISTS paimon_catalog.`default`.cities2
✅ Cleanup complete


## Step 2: Create table WITHOUT Iceberg compatibility

In [18]:
create_sql = f"""
    CREATE TABLE {table_name} (
        country STRING,
        name STRING
    )
"""
print_query(create_sql.strip(), "Creating table WITHOUT Iceberg compatibility")
spark.sql(create_sql)
print("✅ Table created")
print("📚 No 'metadata.iceberg.storage' property set")


   📝 Creating table WITHOUT Iceberg compatibility
   SQL: CREATE TABLE paimon_catalog.`default`.cities2 (
        country STRING,
        name STRING
    )
✅ Table created
📚 No 'metadata.iceberg.storage' property set


## Step 3: Insert initial data (before ALTER)

In [19]:
insert_sql = f"""
    INSERT INTO {table_name} VALUES 
        ('usa', 'new york'),
        ('germany', 'berlin'),
        ('usa', 'chicago'),
        ('germany', 'hamburg')
"""
print_query(insert_sql.strip(), "Inserting 4 rows BEFORE ALTER")
spark.sql(insert_sql)
print("✅ Initial data inserted")

query_before = f"SELECT * FROM {table_name}"
print_query(query_before, "Verifying data via Paimon catalog")
spark.sql(query_before).show()


   📝 Inserting 4 rows BEFORE ALTER
   SQL: INSERT INTO paimon_catalog.`default`.cities2 VALUES 
        ('usa', 'new york'),
        ('germany', 'berlin'),
        ('usa', 'chicago'),
        ('germany', 'hamburg')
✅ Initial data inserted

   📝 Verifying data via Paimon catalog
   SQL: SELECT * FROM paimon_catalog.`default`.cities2
+-------+--------+
|country|    name|
+-------+--------+
|    usa|new york|
|germany|  berlin|
|    usa| chicago|
|germany| hamburg|
+-------+--------+



## Step 4: ALTER TABLE to add Iceberg compatibility

In [20]:
alter_sql = f"""
    ALTER TABLE {table_name}
    SET TBLPROPERTIES ('metadata.iceberg.storage' = 'hadoop-catalog')
"""
print_query(alter_sql.strip(), "Adding Iceberg property via ALTER TABLE")
spark.sql(alter_sql)
print("✅ Property added")
print("📚 Now testing: Is the OLD data visible via Iceberg?")


   📝 Adding Iceberg property via ALTER TABLE
   SQL: ALTER TABLE paimon_catalog.`default`.cities2
    SET TBLPROPERTIES ('metadata.iceberg.storage' = 'hadoop-catalog')
✅ Property added
📚 Now testing: Is the OLD data visible via Iceberg?


## Step 4.1: Query via Iceberg catalog (immediately after ALTER)

In [21]:
query_iceberg = """
    SELECT * FROM iceberg_catalog.`default`.cities2 
    ORDER BY country, name
"""
print_query(query_iceberg.strip(), "Querying via Iceberg catalog")
try:
    spark.sql(query_iceberg).show()
    print("✅ Query succeeded! Old data is visible via Iceberg")
except Exception as e:
    print("⚠️  Could not query via Iceberg:")
    print(f"   Error: {str(e)}")


   📝 Querying via Iceberg catalog
   SQL: SELECT * FROM iceberg_catalog.`default`.cities2 
    ORDER BY country, name
⚠️  Could not query via Iceberg:
   Error: [TABLE_OR_VIEW_NOT_FOUND] The table or view `iceberg_catalog`.`default`.`cities2` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 2 pos 18;
'Sort ['country ASC NULLS FIRST, 'name ASC NULLS FIRST], true
+- 'Project [*]
   +- 'UnresolvedRelation [iceberg_catalog, default, cities2], [], false



## Step 5: Insert NEW data AFTER ALTER

In [22]:
insert_new_sql = f"""
    INSERT INTO {table_name} VALUES 
        ('france', 'paris'),
        ('spain', 'madrid')
"""
print_query(insert_new_sql.strip(), "Inserting 2 new rows AFTER ALTER")
spark.sql(insert_new_sql)
print("✅ New data inserted")


   📝 Inserting 2 new rows AFTER ALTER
   SQL: INSERT INTO paimon_catalog.`default`.cities2 VALUES 
        ('france', 'paris'),
        ('spain', 'madrid')
✅ New data inserted


## Step 6: Query via Iceberg catalog after new data insert

In [23]:
query_iceberg = """
    SELECT * FROM iceberg_catalog.`default`.cities2 
    ORDER BY country, name
"""
print_query(query_iceberg.strip(), "Querying ALL data via Iceberg catalog")

try:
    result = spark.sql(query_iceberg)
    result.show()
    row_count = result.count()
    print(f"✅ Query succeeded! Found {row_count} rows via Iceberg")
    
    # Analyze what data is visible
    print("\n📊 Analysis: Which data is visible via Iceberg?")
    query_old = """
        SELECT * FROM iceberg_catalog.`default`.cities2 
        WHERE country IN ('usa', 'germany')
    """
    print_query(query_old.strip(), "Counting OLD data (usa, germany)")
    old_data = spark.sql(query_old).count()
    
    query_new = """
        SELECT * FROM iceberg_catalog.`default`.cities2 
        WHERE country IN ('france', 'spain')
    """
    print_query(query_new.strip(), "Counting NEW data (france, spain)")
    new_data = spark.sql(query_new).count()
    
    print(f"   • Old data (before ALTER): {old_data} rows")
    print(f"   • New data (after ALTER): {new_data} rows")
    
    # Draw conclusions
    print("\n📚 CONCLUSION:")
    if new_data > 0 and old_data == 0:
        print("   ⚠️  ALTER TABLE enables Iceberg for FUTURE writes only!")
        print("   • Old data (before ALTER) is NOT accessible via Iceberg")
        print("   • New data (after ALTER) IS accessible via Iceberg")
        print("   💡 Recommendation: Set property at CREATE TABLE time")
    elif new_data > 0 and old_data > 0:
        print("   ✅ ALTER TABLE WORKS! All data is accessible via Iceberg!")
        print("   • Both old and new data became accessible after ALTER")
        print("   • Paimon generates Iceberg metadata on-demand")
    else:
        print("   ⚠️  ALTER TABLE did not fully enable Iceberg compatibility")
        
except Exception as e:
    error_msg = str(e)
    print(f"❌ Query failed: {error_msg[:150]}...")
    print("\n📚 CONCLUSION:")
    print("   • ALTER TABLE behavior may be inconsistent")
    print("   💡 Recommendation: Set 'metadata.iceberg.storage' at CREATE TABLE time")


   📝 Querying ALL data via Iceberg catalog
   SQL: SELECT * FROM iceberg_catalog.`default`.cities2 
    ORDER BY country, name
+-------+--------+
|country|    name|
+-------+--------+
| france|   paris|
|germany|  berlin|
|germany| hamburg|
|  spain|  madrid|
|    usa| chicago|
|    usa|new york|
+-------+--------+

✅ Query succeeded! Found 6 rows via Iceberg

📊 Analysis: Which data is visible via Iceberg?

   📝 Counting OLD data (usa, germany)
   SQL: SELECT * FROM iceberg_catalog.`default`.cities2 
        WHERE country IN ('usa', 'germany')

   📝 Counting NEW data (france, spain)
   SQL: SELECT * FROM iceberg_catalog.`default`.cities2 
        WHERE country IN ('france', 'spain')
   • Old data (before ALTER): 4 rows
   • New data (after ALTER): 2 rows

📚 CONCLUSION:
   ✅ ALTER TABLE WORKS! All data is accessible via Iceberg!
   • Both old and new data became accessible after ALTER
   • Paimon generates Iceberg metadata on-demand


---
# DEMO 4: REST Catalog Integration (Optional)

## Advanced Feature: Centralized metadata management
- Paimon registers tables in an Iceberg REST Catalog
- Other tools can discover tables via the REST API
- Enables enterprise-grade metadata management

**Prerequisite**: Docker container must be running
```bash
docker-compose up -d iceberg-rest-catalog
```

In [24]:
# Check if REST catalog is available
rest_catalog_uri = os.getenv('ICEBERG_REST_URI', 'http://localhost:8181')
rest_catalog_name = "iceberg_rest_catalog"

print(f"⚙️  Configuration:")
print(f"   REST Catalog Name: {rest_catalog_name}")
print(f"   REST Catalog URI: {rest_catalog_uri}")
print(f"   Prerequisite: Docker container must be running")

⚙️  Configuration:
   REST Catalog Name: iceberg_rest_catalog
   REST Catalog URI: http://localhost:8181
   Prerequisite: Docker container must be running


In [25]:
def create_spark_session_with_rest_catalog(base_dir, rest_catalog_name, rest_catalog_uri):
    """Create Spark session with REST catalog."""
    paimon_jar, iceberg_jar, paimon_iceberg_jar = find_jar_files(base_dir)
    
    paimon_warehouse = f"file://{base_dir}/warehouse/paimon"
    
    # Build JAR list
    jars_list = [paimon_jar, iceberg_jar]
    if paimon_iceberg_jar:
        jars_list.append(paimon_iceberg_jar)
    jars_string = ",".join(jars_list)
    
    spark = SparkSession.builder \
        .appName("REST Catalog Demo") \
        .config("spark.jars", jars_string) \
        .config("spark.sql.catalog.paimon_catalog", "org.apache.paimon.spark.SparkCatalog") \
        .config("spark.sql.catalog.paimon_catalog.warehouse", paimon_warehouse) \
        .config(f"spark.sql.catalog.{rest_catalog_name}", "org.apache.iceberg.spark.SparkCatalog") \
        .config(f"spark.sql.catalog.{rest_catalog_name}.type", "rest") \
        .config(f"spark.sql.catalog.{rest_catalog_name}.uri", rest_catalog_uri) \
        .config("spark.sql.extensions",
                "org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions,"
                "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("FATAL")
    return spark

In [26]:
# Stop the current session and create one with REST catalog
spark.stop()
spark = create_spark_session_with_rest_catalog(base_dir, rest_catalog_name, rest_catalog_uri)
print("✅ Spark session created with REST catalog")

✅ Spark session created with REST catalog


In [27]:
# Create table with REST catalog integration
table_name = "paimon_catalog.`default`.rest_test"
iceberg_rest_table = f"{rest_catalog_name}.`default`.rest_test"

# Clean up
cleanup_sql = f"DROP TABLE IF EXISTS {table_name}"
print_query(cleanup_sql, "Dropping table if exists")
spark.sql(cleanup_sql)

# Create table with REST catalog integration
create_table_sql = f"""
    CREATE TABLE {table_name} (
        id INT,
        name STRING,
        department STRING
    )
    TBLPROPERTIES (
        'metadata.iceberg.storage' = 'rest-catalog',
        'metadata.iceberg.rest.uri' = '{rest_catalog_uri}',
        'metadata.iceberg.rest.warehouse' = 'paimon_warehouse',
        'metadata.iceberg.rest.clients' = '1'
    )
"""
print_query(create_table_sql.strip(), "Creating table with REST catalog integration")

try:
    spark.sql(create_table_sql)
    print("✅ Table created with REST catalog integration")
    
    # Insert data
    insert_sql = f"""
        INSERT INTO {table_name} VALUES 
            (1, 'Alice', 'Engineering'),
            (2, 'Bob', 'Sales'),
            (3, 'Charlie', 'Marketing')
    """
    print_query(insert_sql.strip(), "Inserting 3 rows via Paimon catalog")
    spark.sql(insert_sql)
    print("✅ Data inserted")
    
    # Query via Paimon
    query_paimon = f"SELECT * FROM {table_name} ORDER BY id"
    print_query(query_paimon, "Querying via Paimon catalog")
    spark.sql(query_paimon).show()
    
    # Query via REST catalog
    query_rest = f"SELECT * FROM {iceberg_rest_table} ORDER BY id"
    print_query(query_rest, "Querying via Iceberg REST catalog")
    result = spark.sql(query_rest)
    result.show()
    print("✅ SUCCESS! Table is accessible via REST Catalog!")
    
except Exception as e:
    print(f"❌ Failed: {str(e)[:200]}...")
    print("\n💡 Possible issues:")
    print("   • REST catalog server might not be running")
    print("   • Start with: docker-compose up -d iceberg-rest-catalog")
    print("   • Requires: paimon-iceberg-1.3.0.jar dependency")


   📝 Dropping table if exists
   SQL: DROP TABLE IF EXISTS paimon_catalog.`default`.rest_test

   📝 Creating table with REST catalog integration
   SQL: CREATE TABLE paimon_catalog.`default`.rest_test (
        id INT,
        name STRING,
        department STRING
    )
    TBLPROPERTIES (
        'metadata.iceberg.storage' = 'rest-catalog',
        'metadata.iceberg.rest.uri' = 'http://localhost:8181',
        'metadata.iceberg.rest.warehouse' = 'paimon_warehouse',
        'metadata.iceberg.rest.clients' = '1'
    )
✅ Table created with REST catalog integration

   📝 Inserting 3 rows via Paimon catalog
   SQL: INSERT INTO paimon_catalog.`default`.rest_test VALUES 
            (1, 'Alice', 'Engineering'),
            (2, 'Bob', 'Sales'),
            (3, 'Charlie', 'Marketing')
✅ Data inserted

   📝 Querying via Paimon catalog
   SQL: SELECT * FROM paimon_catalog.`default`.rest_test ORDER BY id
+---+-------+-----------+
| id|   name| department|
+---+-------+-----------+
|  1|  Alice|

In [28]:
# Cleanup REST demo
try:
    drop_rest_sql = f"DROP TABLE IF EXISTS {iceberg_rest_table}"
    spark.sql(drop_rest_sql)
except:
    pass

drop_sql = f"DROP TABLE IF EXISTS {table_name}"
spark.sql(drop_sql)
print("✅ REST de   mo cleanup complete")

✅ REST de   mo cleanup complete


---
# Key Takeaways

1. **Paimon tables can be read by Iceberg tools**
2. **Use `'metadata.iceberg.storage' = 'hadoop-catalog'`**
3. **Set the property at CREATE TABLE time (recommended)**
4. **REST Catalog enables centralized metadata management**

## Best Practices
- Set Iceberg compatibility at table creation time
- Manually drop both Paimon and Iceberg tables when cleaning up
- Use REST Catalog for enterprise metadata management

---
# Cleanup: Stop Spark session

In [29]:
spark.stop()
print("🛑 Spark session stopped")

🛑 Spark session stopped
